# Fig. S7D: Gene expression changes associated with Hepatovirus infection in chicken liver samples, assessed by GSEA

Run GSEA on results from `FigS7D_DESeq2.R` and prepare tables for `FigS7D_plot.R`.


In [1]:
import pandas as pd
import gseapy as gp
from gseapy import Msigdb
import os
import numpy as np

In [2]:
# Define paths
basedir = "/path/to/project"
directory_out = "FigS7D/gsea"

# Output directory
path_out = f"{basedir}/output/revise_final/{directory_out}"

# Input directory
input_folder_path = f"{basedir}/output/revise_final/FigS7D/deseq2"

os.makedirs(path_out, exist_ok=True)
print("saving files to:", path_out)


saving files to: /Users/coco/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/revise_final/FigS7D/gsea


In [3]:
# Load the Hallmark gene sets
gene_sets = gp.get_library(
    name="MSigDB_Hallmark_2020",
    organism="Human"
)

# Load the MSigDB C2:CGP gene sets
msig = Msigdb()

c2_cgp = msig.get_gmt(
    category="c2.cgp",
    dbver="2026.1.Hs"
)

# Add the WIELAND_UP_BY_HBV_INFECTION gene set
gene_sets["WIELAND_UP_BY_HBV_INFECTION"] = \
    c2_cgp["WIELAND_UP_BY_HBV_INFECTION"]


In [4]:
def plot_prerank(df, outdir_path, metric, gene_col, gene_sets):
    """
    Run preranked GSEA and plot the top five terms.

    metric: Column used to rank genes.
    gene_col: Column containing gene symbols.
    """
    # Input columns: gene_name_hs, gene_name_gallus, baseMean, log2FoldChange, lfcSE, stat, pvalue, padj
    df2 = df.copy()
    
    # Remove rows with missing gene names or ranking values
    df2 = df2.dropna(subset=[gene_col, metric])
    df2[gene_col] = df2[gene_col].astype(str)
    
    # For duplicate genes, keep the largest absolute ranking value
    # Break ties by p-value, then gene name
    df2["_abs"] = df2[metric].astype(float).abs()
    df2["_p"] = df2["pvalue"].astype(float).fillna(1.0)

    df2 = (df2.sort_values([gene_col, "_abs", "_p", gene_col],
                           ascending=[True, False, True, True],
                           kind="mergesort").drop_duplicates(subset=[gene_col], keep="first"))

    # Rank genes, breaking ties by p-value and gene name
    df2 = df2.sort_values([metric, "_p", gene_col],
                          ascending=[False, True, True],
                          kind="mergesort")
    rnk = df2[[gene_col, metric]]
    print("\nrnk\n", rnk.head(3))

    # Run preranked GSEA
    pre_res = gp.prerank(
        rnk=rnk,
        gene_sets=gene_sets,
        outdir=f"{outdir_path}_wieland_hbv_{metric}_{gene_col}",
        seed=0,
        threads=4,
        verbose=True
    )

    # Plot the top five terms
    terms = pre_res.res2d["Term"]
    pre_res.plot(
        terms=terms[:5], 
        legend_kws={'loc': (1.15, 0)},
        ofname=f"{outdir_path}_wieland_hbv_{metric}_{gene_col}_top5_enriched.png"
    )

In [5]:
configs = {
    "metric": "stat",
    "gene_col": "gene_name_hs",
    "gene_sets": gene_sets
}

for dirpath, dirnames, filenames in os.walk(input_folder_path):
    for fname in filenames:
        if not(fname.endswith("_hs_gallus_and_Gene.txt")):
            continue

        # Read the DESeq2 results
        path = os.path.join(dirpath, fname)
        project = pd.read_table(path)

        # Run and plot preranked GSEA
        proname = fname.split("_")[1]
        print("\nprocessing...", proname)
        plot_prerank(project, f"{path_out}/{proname}", configs["metric"], configs["gene_col"], configs["gene_sets"])

2026-11-20 20:27:30,388 [WARNING] Duplicated values found in preranked stats: 3.96% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-11-20 20:27:30,388 [INFO] Parsing data files for GSEA.............................
2026-11-20 20:27:30,397 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=15
2026-11-20 20:27:30,398 [INFO] 0051 gene_sets used for further statistical testing.....
2026-11-20 20:27:30,398 [INFO] Start to run GSEA...Might take a while..................



processing... PRJNA774885

rnk
       gene_name_hs      stat
9514         SMYD1  9.237524
19233          PI3  8.965137
4314          TGM4  8.192610


2026-11-20 20:27:34,867 [INFO] Congratulations. GSEApy runs successfully................

2026-11-20 20:27:35,116 [WARNING] Duplicated values found in preranked stats: 1.42% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-11-20 20:27:35,116 [INFO] Parsing data files for GSEA.............................
2026-11-20 20:27:35,127 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=15
2026-11-20 20:27:35,128 [INFO] 0051 gene_sets used for further statistical testing.....
2026-11-20 20:27:35,128 [INFO] Start to run GSEA...Might take a while..................



processing... PRJEB63475

rnk
      gene_name_hs       stat
4515         BRD9  15.148027
212          UPF2  14.855553
4021        PHF14  12.613871


2026-11-20 20:27:39,934 [INFO] Congratulations. GSEApy runs successfully................



In [6]:
# Select significant terms among the top and bottom ten NES values
terms = set()

for dirpath, dirnames, filenames in os.walk(path_out):
    for fname in filenames:
        if fname != "gseapy.gene_set.prerank.report.csv":
            continue

        # Read the GSEA results
        path = os.path.join(dirpath, fname)
        res = pd.read_csv(path)
        project = path.split("/")[-2].split("_")[0]

        # Select the ten highest and ten lowest NES values
        top10   = res.sort_values("NES", ascending=False).head(10).copy()
        worst10 = res.sort_values("NES", ascending=True).head(10).copy()
        df = pd.concat([top10, worst10], ignore_index=True).sort_values("NES", ascending=True).reset_index(drop=True)

        # Collect terms with FDR < 0.05
        for t in df.loc[df["FDR q-val"] < 0.05, "Term"].to_list():
            terms.add(t)

# Check the number of selected terms
print("top 10 and FDR<0.05:", len(terms)) 

# Sort the selected terms
terms = sorted(list(terms))

top 10 and FDR<0.05: 27


In [7]:
# Order terms by mean NES
first_file = True

for dirpath, dirnames, filenames in os.walk(path_out):
    for fname in filenames:
        if fname != "gseapy.gene_set.prerank.report.csv":
            continue

        # Read the GSEA results
        path = os.path.join(dirpath, fname)
        res = pd.read_csv(path)
        project = path.split("/")[-2].split("_")[0]

        # Keep the selected terms
        df_term = res.loc[res["Term"].isin(terms), ["Term", "NES"]].copy()
        df_term.columns = df_term.columns.map(lambda x: f"{x}_{project}" if x != "Term" else x)
        print(f"Project: {project} size: {df_term.shape}")

        # Merge results across projects
        if first_file:
            df_all = df_term
            first_file = False
        else:
            df_all = df_all.merge(df_term, on="Term", how="left")

# Check dimensions
print("df_all size:", df_all.shape)

# Sort terms by mean NES
cols = df_all.set_index("Term").columns.to_list()

df_all["mean_NES"] = df_all[cols].mean(axis=1)
df_sorted = df_all.sort_values("mean_NES", ascending=True)

# Store the term order
terms_order = df_sorted["Term"].to_list()

# Preview results
print(terms_order)

Project: PRJNA774885 size: (27, 2)
Project: PRJEB63475 size: (27, 2)
df_all size: (27, 3)
['TGF-beta Signaling', 'Epithelial Mesenchymal Transition', 'Estrogen Response Late', 'Hypoxia', 'Apical Junction', 'UV Response Dn', 'KRAS Signaling Dn', 'Oxidative Phosphorylation', 'Pperoxisome', 'Adipogenesis', 'Fatty Acid Metabolism', 'Myc Targets V1', 'Myogenesis', 'Wnt-beta Catenin Signaling', 'Apical Surface', 'mTORC1 Signaling', 'Reactive Oxygen Species Pathway', 'Xenobiotic Metabolism', 'Coagulation', 'Protein Secretion', 'WIELAND_UP_BY_HBV_INFECTION', 'Complement', 'Inflammatory Response', 'IL-6/JAK/STAT3 Signaling', 'Allograft Rejection', 'Interferon Alpha Response', 'Interferon Gamma Response']


In [8]:
# Create the dot plot table
first_file=True
dfs = []

for dirpath, dirnames, filenames in os.walk(path_out):
    for fname in filenames:
        if fname != "gseapy.gene_set.prerank.report.csv":
            continue

        # Read the GSEA results
        path = os.path.join(dirpath, fname)
        res = pd.read_csv(path)
        project = path.split("/")[-2].split("_")[0]

        # Calculate −log10(FDR), clipping FDR at 1e-10
        res["mlog10_fdr"] = -np.log10(res["FDR q-val"].clip(lower=1e-10))

        # Keep the selected terms
        df_term = res.loc[res["Term"].isin(terms), ["Term", "NES", "FDR q-val", "mlog10_fdr"]].copy()
        df_term["Project"] = project
        df_term_sorted = df_term.sort_values(
            "Term",
            key=lambda s: pd.Categorical(s, categories=terms_order, ordered=True)
        )

        dfs.append(df_term_sorted)

# Combine results
df_cat = pd.concat(dfs, ignore_index=True)
# Save tables
df_cat.to_csv(f"{path_out}/NES_top10_FDR05.csv")
# Preview results
print("df_cat size:", df_cat.shape)
df_cat.head(3)

df_cat size: (54, 5)


,Term,NES,FDR q-val,mlog10_fdr,Project
0,TGF-beta Signaling,-1.497409,0.031990,1.494983,PRJNA774885
1,Epithelial Mesenchymal Transition,-1.923752,0.004667,2.330938,PRJNA774885
2,Estrogen Response Late,-1.614328,0.021003,1.677725,PRJNA774885


In [9]:
# Create a PRJNA774885-only table from the existing GSEA report (no GSEA rerun)
one_project_id = "PRJNA774885"
one_project_report_path = os.path.join(
    path_out,
    f"{one_project_id}_wieland_hbv_{configs['metric']}_{configs['gene_col']}",
    "gseapy.gene_set.prerank.report.csv"
)
one_project_res = pd.read_csv(one_project_report_path)

# Same selection rule: NES top/bottom 10, followed by FDR < 0.05
one_project_top10 = one_project_res.sort_values("NES", ascending=False).head(10)
one_project_worst10 = one_project_res.sort_values("NES", ascending=True).head(10)
one_project_df = pd.concat(
    [one_project_top10, one_project_worst10], ignore_index=True
).drop_duplicates(subset=["Term"])
one_project_df = one_project_df.loc[
    one_project_df["FDR q-val"] < 0.05, ["Term", "NES", "FDR q-val"]
].copy()
one_project_df["mlog10_fdr"] = -np.log10(
    one_project_df["FDR q-val"].clip(lower=1e-10)
)
one_project_df["Project"] = one_project_id
one_project_df = one_project_df.sort_values("NES", ascending=True).reset_index(drop=True)

one_project_output_path = os.path.join(path_out, "NES_top10_FDR05_one_project.csv")
one_project_df.to_csv(one_project_output_path, index=False)
print(f"{one_project_id}: {len(one_project_df)} terms; saved to {one_project_output_path}")
one_project_df.head(3)

PRJNA774885: 20 terms; saved to /Users/coco/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/revise_final/FigS7D/gsea/NES_top10_FDR05_one_project.csv


,Term,NES,FDR q-val,mlog10_fdr,Project
0,Epithelial Mesenchymal Transition,-1.923752,0.004667,2.330938,PRJNA774885
1,Hypoxia,-1.669719,0.018669,1.728878,PRJNA774885
2,Myogenesis,-1.615299,0.028004,1.552786,PRJNA774885
